# 10 Model comparison on G0–G3

All six models side by side, read from `results/log.csv` (validation rows of the G0–G3 runs from notebooks 04 to
09). A cell is shown only when its model has been validated with all of seeds 0, 1 and 2 on that set; the first
table lists what is still missing. Nothing is refitted here.

Sections 3 and 4 split the scores into **known** and **unknown** customers (a `uid` already seen in the data the
model was trained on, or not), using the scores saved in `results/preds/`. The split uses only information available
when a transaction arrives, so it can also be used to weight models in an ensemble without leakage.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from src import experiment, feature_sets as fs, segments
from src.metrics import top_rows
from src.split import holdout_indices

MODELS = ["logreg", "rf", "lgbm", "xgboost", "catboost", "mlp"]
LABELS = {"logreg": "Logistic regression", "rf": "Random forest", "lgbm": "LightGBM",
          "xgboost": "XGBoost", "catboost": "CatBoost", "mlp": "MLP"}
FEATURE_SETS = list(fs.FEATURE_SETS)
SEEDS = [0, 1, 2]
METRICS = {"pr_auc": "PR-AUC", "auroc": "AUROC", "recall_top1pct": "recall in top 1%",
           "precision_top1pct": "precision in top 1%", "precision": "precision at threshold",
           "recall": "recall at threshold"}
pd.set_option("display.width", 160)

## 1. What has been run

In [ ]:
spec = fs.load_spec()
ids = pd.read_parquet(ROOT / spec["parquet"],
                      columns=["TransactionID", "TransactionDT", "TransactionDay", "uid", "isFraud"])
train_idx, val_idx, _ = holdout_indices(ids)
ids = ids.iloc[:val_idx[-1] + 1]                       # the test block is not used
sizes = experiment.block_sizes(ids, train_idx, val_idx)
n_val, fraud_val = sizes["val"]

log = pd.read_csv(experiment.LOG_PATH)
val = log[(log["split"] == "val") & log["model"].isin(MODELS) & log["feature_set"].isin(FEATURE_SETS)].copy()

def settings(p):
    d = json.loads(p)
    return json.dumps({k: v for k, v in d.items() if k not in ("threshold", "_fit")}, sort_keys=True)

val["settings"] = val["params"].map(settings)
# precision in the top 1%, derived from the logged recall: recall x fraud rows / rows flagged
val["precision_top1pct"] = val["recall_top1pct"] * fraud_val / top_rows(n_val)
status, rows = {}, []
for m in MODELS:
    for g in FEATURE_SETS:
        cell = val[(val["model"] == m) & (val["feature_set"] == g)]
        complete = [s for s, part in cell.groupby("settings") if sorted(part["seed"]) == SEEDS]
        if len(complete) > 1:
            raise ValueError(f"{m}/{g}: more than one set of settings validated with all seeds; "
                             "keep one in the log or choose explicitly")
        status[(m, g)] = "done" if complete else (f"seeds {sorted(cell['seed'])}" if len(cell) else "not run")
        if complete:
            rows.append(cell[cell["settings"] == complete[0]])
done = pd.concat(rows) if rows else val.iloc[:0]
pd.Series(status).unstack().reindex(index=MODELS, columns=FEATURE_SETS).rename(index=LABELS)

## 2. Validation results (mean ± sample sd over seeds 0, 1, 2)

Precision in the top 1% is derived from the logged recall in the top 1% and ranks models the same way; recall in the top
1% is capped well below 1 (printed below). Precision and recall at the threshold use the F1-best threshold picked on
the validation block itself, so they are slightly optimistic.

In [ ]:
def table(metric):
    g = done.groupby(["model", "feature_set"])[metric]
    cell = g.mean().map("{:.4f}".format) + " ± " + g.std().map("{:.4f}".format)
    return cell.unstack().reindex(index=MODELS, columns=FEATURE_SETS).rename(index=LABELS)

print(f"Validation block: {fraud_val:,} fraud in {n_val:,} rows; the top 1% is {top_rows(n_val):,} rows, "
      f"so recall in the top 1% is at most {top_rows(n_val) / fraud_val:.3f}.")
for metric, label in METRICS.items():
    print(label)
    display(table(metric))

In [ ]:
mean = done.groupby(["model", "feature_set"])["pr_auc"].mean().unstack().reindex(index=MODELS, columns=FEATURE_SETS)
sd = done.groupby(["model", "feature_set"])["pr_auc"].std().unstack().reindex(index=MODELS, columns=FEATURE_SETS)
fig, ax = plt.subplots(figsize=(10, 4))
width = 0.8 / len(FEATURE_SETS)
x = np.arange(len(MODELS))
for i, g in enumerate(FEATURE_SETS):
    ax.bar(x + i * width, mean[g], width, yerr=sd[g].fillna(0), capsize=2, label=g)
ax.set_xticks(x + width * (len(FEATURE_SETS) - 1) / 2, [LABELS[m] for m in MODELS])
ax.set_ylabel("validation PR-AUC")
ax.set_title("Validation PR-AUC by model and feature set (mean ± sd over seeds)")
ax.legend(title="feature set")
ax.grid(axis="x")
fig.tight_layout()
plt.show()

**What each feature group adds:** the change in mean validation PR-AUC from one set to the next, per model.

In [ ]:
gain = mean.diff(axis=1).iloc[:, 1:]
gain.columns = [f"{a} → {b}" for a, b in zip(FEATURE_SETS[:-1], FEATURE_SETS[1:])]
gain["G0 → G3"] = mean["G3"] - mean["G0"]
gain.rename(index=LABELS).round(4)

## 3. Known vs unknown customers (validation block)

A validation row is **known** if its `uid` appears in the training block, otherwise **unknown**
(`src/segments.py`). History features (G2, G3) can only help known customers, so comparing the two segments shows
where the customer profile pays off, and whether different models are best for different segments, which is the
case for an ensemble. PR-AUC is the mean over seeds 0, 1, 2; the interval is a 95% bootstrap interval over rows
(`N_BOOT` resamples). Overlapping intervals mean the gap between two models may be noise.

In [ ]:
known_by_id = pd.Series(segments.known_uid(ids["uid"], set(ids["uid"].iloc[train_idx])),
                        index=ids["TransactionID"].to_numpy())
print(f"validation rows: known {known_by_id.iloc[val_idx].mean():.1%}, "
      f"unknown {1 - known_by_id.iloc[val_idx].mean():.1%}")

N_BOOT = 200
rows = []
for m in MODELS:
    for g in FEATURE_SETS:
        paths = [experiment.PREDS_DIR / f"{m}_{g}_val_seed{s}.parquet" for s in SEEDS]
        if status[(m, g)] != "done" or not all(p.exists() for p in paths):
            continue
        preds = [pd.read_parquet(p) for p in paths]
        if not all(p["TransactionID"].equals(preds[0]["TransactionID"]) for p in preds):
            raise ValueError(f"{m}/{g}: saved validation scores cover different rows")
        known = known_by_id.reindex(preds[0]["TransactionID"]).to_numpy()
        t = segments.segment_pr_auc(preds[0]["isFraud"].to_numpy(), [p["score"].to_numpy() for p in preds],
                                    known, n_boot=N_BOOT)
        rows.append(t.assign(model=m, feature_set=g))
by_segment = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

if len(by_segment):
    display(by_segment.groupby("segment")[["rows", "fraud"]].first())
    text = (by_segment["pr_auc"].map("{:.4f}".format) + " [" + by_segment["low"].map("{:.3f}".format)
            + ", " + by_segment["high"].map("{:.3f}".format) + "]")
    table_seg = (by_segment.assign(text=text).pivot(index="model", columns=["segment", "feature_set"], values="text")
                 .reindex([m for m in MODELS if m in set(by_segment["model"])]).rename(index=LABELS))
    display(table_seg)
else:
    print("No complete validation scores saved yet: run notebooks 04 to 09 first.")

In [ ]:
if len(by_segment):
    g = [g for g in FEATURE_SETS if g in set(by_segment["feature_set"])][-1]   # the largest set with results
    t = by_segment[by_segment["feature_set"] == g]
    models = [m for m in MODELS if m in set(t["model"])]
    fig, ax = plt.subplots(figsize=(9, 3.8))
    x = np.arange(len(models))
    for i, seg in enumerate(segments.SEGMENTS):
        part = t[t["segment"] == seg].set_index("model").loc[models]
        err = np.vstack([part["pr_auc"] - part["low"], part["high"] - part["pr_auc"]])
        ax.bar(x + (i - 0.5) * 0.38, part["pr_auc"], 0.38, yerr=err, capsize=3, label=f"{seg} uid")
    ax.set_xticks(x, [LABELS[m] for m in models])
    ax.set_ylabel("validation PR-AUC")
    ax.set_title(f"Known vs unknown customers, {g} (mean over seeds, 95% bootstrap interval)")
    ax.legend()
    ax.grid(axis="x")
    fig.tight_layout()
    plt.show()

## 4. Known vs unknown on the time-CV fold scores (for an ensemble)

To choose ensemble weights without using the validation block, use the fold scores saved by the model notebooks
(`SAVE_OOF`). Within each fold, **known** means the `uid` appears in that fold's training months
(`segments.fold_reference_uids`), matching what the model saw. PR-AUC is computed per fold and averaged over the
4 folds. The validation block (section 3) then checks whatever is chosen here.

In [ ]:
fold_ref = segments.fold_reference_uids(ids, train_idx)
uid_by_id = pd.Series(ids["uid"].to_numpy(), index=ids["TransactionID"].to_numpy())
rows = []
for m in MODELS:
    for g in FEATURE_SETS:
        path = experiment.PREDS_DIR / f"{m}_{g}_oof_seed0.parquet"
        if not path.exists():
            continue
        oof = pd.read_parquet(path)
        uid = uid_by_id.reindex(oof["TransactionID"]).to_numpy()
        known = np.array([u in fold_ref[k] for u, k in zip(uid, oof["fold"])])
        for k, part in oof.groupby("fold"):
            mask = (oof["fold"] == k).to_numpy()
            t = segments.segment_pr_auc(part["isFraud"].to_numpy(), part["score"].to_numpy(), known[mask], n_boot=0)
            rows.append(t.assign(model=m, feature_set=g, fold=k))
oof_segment = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
if len(oof_segment):
    print("Mean PR-AUC over the 4 folds; known share of fold-validation rows:",
          f"{oof_segment.groupby('segment')['rows'].sum()['known'] / oof_segment['rows'].sum():.1%}")
    display(oof_segment.groupby(["model", "segment", "feature_set"])["pr_auc"].mean().unstack(["segment", "feature_set"])
            .reindex([m for m in MODELS if m in set(oof_segment["model"])]).rename(index=LABELS).round(4))
else:
    print("No fold scores saved yet: run notebooks 04 to 09 with SAVE_OOF = True.")

## 5. Findings

Write this once all cells are done, quoting only numbers from the tables above (they come from `results/log.csv`
and the saved scores):

- Which model and feature set score best on validation PR-AUC, and by how much over the next best.
- How much each feature group adds, and whether that is the same for every model.
- How known and unknown customers differ, and whether a different model is best for each segment (with intervals
  that don't overlap). If so, an ensemble weighted by segment is worth trying, with the weights chosen on the
  fold scores (section 4) and checked on validation (section 3).